### Architecture
```
Bronze CDC
    │
    │ account_id + changed_column + new_value
    ▼
Prepare latest changes
    │
    ▼
Pivot column-level changes
    │
    ▼
MERGE into Silver Accounts
    │
    ├── existing account → UPDATE
    └── new account      → INSERT

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CDC_TABLE = "dbx_fintech_data_platform.bronze.account_cdc"
ACCOUNT_TABLE = "dbx_fintech_data_platform.silver.accounts"

In [0]:
accounts_df = spark.table(ACCOUNT_TABLE)

print("Silver account records:", accounts_df.count())

accounts_df.printSchema()

display(
    accounts_df.limit(20)
)

In [0]:
cdc_df = spark.table(CDC_TABLE)

print("CDC records:", cdc_df.count())

display(
    cdc_df.orderBy("change_timestamp").limit(20)
)

In [0]:
display(
    cdc_df.groupBy("change_type")
          .count()
)

In [0]:
display(
    cdc_df.groupBy("changed_column")
          .count()
)

In [0]:
effective_cdc_df = (
    cdc_df
    .filter(F.col("change_type") == "UPDATE")
    .filter(
        F.coalesce(F.col("old_value"), F.lit("")) !=
        F.coalesce(F.col("new_value"), F.lit(""))
    )
)

In [0]:
print(
    "Effective CDC changes:",
    effective_cdc_df.count()
)

In [0]:
window_spec = (
    Window
    .partitionBy("account_id", "changed_column")
    .orderBy(F.col("change_timestamp").desc())
)

latest_cdc_df = (
    effective_cdc_df
    .withColumn("_rn", F.row_number().over(window_spec))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

In [0]:
display(
    latest_cdc_df.orderBy("account_id")
)

In [0]:
pivoted_cdc_df = (
    latest_cdc_df
    .groupBy("account_id")
    .pivot("changed_column", ["account_type", "status"])
    .agg(F.first("new_value"))
)

In [0]:
display(pivoted_cdc_df)

In [0]:
latest_timestamp_df = (
    latest_cdc_df
    .groupBy("account_id")
    .agg(
        F.max("change_timestamp").alias("cdc_change_timestamp")
    )
)

cdc_ready_df = (
    pivoted_cdc_df
    .join(
        latest_timestamp_df,
        on="account_id",
        how="inner"
    )
)

display(cdc_ready_df)

In [0]:
cdc_ready_df.createOrReplaceTempView(
    "account_cdc_ready"
)

In [0]:
%sql

MERGE INTO dbx_fintech_data_platform.silver.accounts AS target

USING account_cdc_ready AS source

ON target.account_id = source.account_id

WHEN MATCHED THEN
UPDATE SET
    target.account_type =
        CASE
            WHEN source.account_type IS NOT NULL
            THEN source.account_type
            ELSE target.account_type
        END,

    target.status =
        CASE
            WHEN source.status IS NOT NULL
            THEN source.status
            ELSE target.status
        END,

    target.updated_at =
        CASE
            WHEN source.cdc_change_timestamp IS NOT NULL
            THEN source.cdc_change_timestamp
            ELSE target.updated_at
        END;

In [0]:
%sql

SELECT
    account_id,
    account_type,
    status,
    updated_at
FROM dbx_fintech_data_platform.silver.accounts
WHERE account_id IN (
    SELECT account_id
    FROM account_cdc_ready
)
LIMIT 20;

In [0]:
%sql

SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT account_id) AS unique_accounts
FROM dbx_fintech_data_platform.silver.accounts;

In [0]:
%sql

SELECT
    account_id,
    COUNT(*) AS record_count
FROM dbx_fintech_data_platform.silver.accounts
GROUP BY account_id
HAVING COUNT(*) > 1;

In [0]:
%sql

SELECT
    c.account_id,
    c.new_value AS expected_account_type,
    a.account_type AS silver_account_type
FROM (
    SELECT
        account_id,
        new_value,
        ROW_NUMBER() OVER (
            PARTITION BY account_id
            ORDER BY change_timestamp DESC
        ) AS rn
    FROM dbx_fintech_data_platform.bronze.account_cdc
    WHERE changed_column = 'account_type'
      AND change_type = 'UPDATE'
      AND old_value <> new_value
) c
JOIN dbx_fintech_data_platform.silver.accounts a
    ON c.account_id = a.account_id
WHERE c.rn = 1
  AND c.new_value <> a.account_type;

In [0]:
%sql

SELECT
    c.account_id,
    c.new_value AS expected_status,
    a.status AS silver_status
FROM (
    SELECT
        account_id,
        new_value,
        ROW_NUMBER() OVER (
            PARTITION BY account_id
            ORDER BY change_timestamp DESC
        ) AS rn
    FROM dbx_fintech_data_platform.bronze.account_cdc
    WHERE changed_column = 'status'
      AND change_type = 'UPDATE'
      AND old_value <> new_value
) c
JOIN dbx_fintech_data_platform.silver.accounts a
    ON c.account_id = a.account_id
WHERE c.rn = 1
  AND c.new_value <> a.status;